# QLoRA Fine-Tuning in Google Colab

This notebook fine-tunes Mistral-7B-Instruct with your capstone dataset using QLoRA.

It will:
1. Verify GPU availability
2. Install required packages
3. Upload `training_data.jsonl` and `fine_tune.py`
4. Run training
5. Zip and download the adapter output

In [1]:
import subprocess
import torch

print('[+] GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('[+] GPU name:', torch.cuda.get_device_name(0))
    print('[+] CUDA version (torch):', torch.version.cuda)
    print('[+] Device count:', torch.cuda.device_count())
else:
    print('[!] No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU')

print('\n[+] nvidia-smi:')
try:
    result = subprocess.run(['nvidia-smi'], check=False, text=True, capture_output=True)
    print(result.stdout if result.stdout else result.stderr)
except Exception as e:
    print('[!] Could not run nvidia-smi:', e)

[+] GPU available: True
[+] GPU name: Tesla T4
[+] CUDA version (torch): 12.8
[+] Device count: 1

[+] nvidia-smi:
Thu Mar 26 12:52:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                        

In [2]:
from pathlib import Path

req_path = Path('requirements_finetune.txt')
if not req_path.exists():
    req_path.write_text(
        '\n'.join([
            'torch>=2.1.0',
            'transformers>=4.40.0',
            'peft>=0.10.0',
            'trl>=0.8.0',
            'bitsandbytes>=0.43.0',
            'datasets>=2.18.0',
            'accelerate>=0.28.0',
        ]) + '\n',
        encoding='utf-8'
    )

!pip install -q -r requirements_finetune.txt
print('[✓] Installed packages from requirements_finetune.txt')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 14.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.8 MB/s eta 0:00:00:00:0100:01
[✓] Installed packages from requirements_finetune.txt


In [4]:
from google.colab import files
import os

print('Upload training_data.jsonl first, then fine_tune.py')
uploaded = files.upload()

for filename in uploaded:
    print(f'[✓] Uploaded: {filename}')

print('\n[+] /content files:')
print(os.listdir('/content'))
print("[i] Expect to see: training_data.jsonl and fine_tune.py")

Upload training_data.jsonl first, then fine_tune.py


KeyboardInterrupt: 

In [3]:
from pathlib import Path

# Resolve files robustly whether notebook is run from /root/capstone or /root/capstone/scenario_generation
cwd = Path.cwd()
candidates = {
    'training_data.jsonl': [
        cwd / 'training_data.jsonl',
        cwd / 'dataset' / 'training_data.jsonl',
        cwd / 'scenario_generation' / 'dataset' / 'training_data.jsonl',
    ],
    'fine_tune.py': [
        cwd / 'fine_tune.py',
        cwd / 'fine_tuning' / 'fine_tune.py',
        cwd / 'scenario_generation' / 'fine_tuning' / 'fine_tune.py',
    ],
}

resolved = {}
missing = []
for name, paths in candidates.items():
    found = next((p for p in paths if p.exists()), None)
    if found is None:
        missing.append(name)
    else:
        resolved[name] = found.resolve()

if missing:
    raise FileNotFoundError(
        f'Missing required files: {missing}\n'
        f'Current working directory: {cwd}\n'
        'Put files in fine_tuning/ or keep repo structure intact.'
    )

data_path = resolved['training_data.jsonl']
script_path = resolved['fine_tune.py']

print(f'[✓] Found training_data.jsonl at: {data_path}')
print(f'[✓] Found fine_tune.py at: {script_path}')

FileNotFoundError: Missing required files: ['training_data.jsonl', 'fine_tune.py']
Current working directory: /content
Put files in fine_tuning/ or keep repo structure intact.

In [ ]:
# Uses data_path and script_path from the previous cell
!python "{script_path}" --data "{data_path}" --out ./output

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('finetuned_adapters', 'zip', './output')
files.download('finetuned_adapters.zip')